# Homework 04 - Data Acquisition and Ingestion

This notebook pulls one dataset from an API, scrapes one public table, validates both datasets, and saves the raw outputs to `data/raw/`.

In [17]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
HOMEWORK_ROOT = Path.cwd().parent
DATA_RAW = HOMEWORK_ROOT / "data" / "raw"

timestamp = datetime.now().strftime("%Y%m%d-%H%M")

print("Homework root:", HOMEWORK_ROOT)
print("Raw data folder:", DATA_RAW)
print("Raw data folder exists:", DATA_RAW.exists())
print("Timestamp:", timestamp)

Homework root: /Users/devampatel/Documents/NYU/homework/homework04
Raw data folder: /Users/devampatel/Documents/NYU/homework/homework04/data/raw
Raw data folder exists: True
Timestamp: 20260820-2242


## 1. API Pull

This section pulls daily Apple stock price data from a public Yahoo Finance chart endpoint. The response is converted into a pandas DataFrame, validated, and saved as a raw CSV file.

In [10]:
base_currency = "USD"
url = "https://api.frankfurter.app/latest"

params = {
    "from": base_currency,
    "to": "EUR,GBP,JPY,CAD"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("Final URL:", response.url)

response.raise_for_status()

Status code: 200
Final URL: https://api.frankfurter.dev/v1/latest?from=USD&to=EUR%2CGBP%2CJPY%2CCAD


In [11]:
data = response.json()

api_df = pd.DataFrame(
    data["rates"].items(),
    columns=["currency", "exchange_rate"]
)

api_df["base_currency"] = data["base"]
api_df["date"] = data["date"]

api_df

,currency,exchange_rate,base_currency,date
0,CAD,1.37700,USD,2026-08-20
1,EUR,0.85609,USD,2026-08-20
2,GBP,0.73388,USD,2026-08-20
3,JPY,158.76000,USD,2026-08-20


### API Validation

The API data is checked for required columns, missing values, row count, and numeric exchange-rate values.

In [12]:
required_api_columns = ["currency", "exchange_rate", "base_currency", "date"]

print("Shape:", api_df.shape)
print("Required columns present:", all(col in api_df.columns for col in required_api_columns))
print("Missing values:")
print(api_df[required_api_columns].isna().sum())

api_df["exchange_rate"] = pd.to_numeric(api_df["exchange_rate"], errors="coerce")

print("Exchange rate dtype:", api_df["exchange_rate"].dtype)
print("Rows with missing exchange rate:", api_df["exchange_rate"].isna().sum())

Shape: (4, 4)
Required columns present: True
Missing values:
currency         0
exchange_rate    0
base_currency    0
date             0
dtype: int64
Exchange rate dtype: float64
Rows with missing exchange rate: 0


In [13]:
api_output_path = DATA_RAW / f"api_frankfurter_exchange_rates_{timestamp}.csv"

api_df.to_csv(api_output_path, index=False)

print("Saved API data to:", api_output_path)

Saved API data to: /Users/devampatel/Documents/NYU/homework/homework04/data/raw/api_frankfurter_exchange_rates_20260820-2242.csv


## 2. Scrape a Small Table

This section scrapes a public Wikipedia table of circulating currencies, validates the table, and saves it as a raw CSV file.

In [20]:
scrape_url = "https://en.wikipedia.org/wiki/List_of_circulating_currencies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

scrape_response = requests.get(scrape_url, headers=headers, timeout=30)

print("Status code:", scrape_response.status_code)
print("Final URL:", scrape_response.url)

scrape_response.raise_for_status()

Status code: 200
Final URL: https://en.wikipedia.org/wiki/List_of_circulating_currencies


In [21]:
soup = BeautifulSoup(scrape_response.text, "html.parser")

table = soup.find("table", class_="wikitable")

headers = []
for header_cell in table.find_all("th"):
    headers.append(header_cell.get_text(strip=True))

rows = []
for row in table.find_all("tr")[1:]:
    cells = row.find_all(["td", "th"])
    row_values = [cell.get_text(strip=True) for cell in cells]
    if row_values:
        rows.append(row_values)

scraped_df = pd.DataFrame(rows)

scraped_df.head()

,0,1,2,3,4,5
0,Abkhazia,Abkhazian apsar[E],аԥ,(none),(none),(none)
1,Russian ruble,₽,RUB,Kopeck,100,None
2,Afghanistan,Afghan afghani,؋‎,AFN,Pul,100
3,Akrotiri and Dhekelia,Euro,€,EUR,Cent,100
4,Albania,Albanian lek,L,ALL,Qintar,100


In [22]:
scraped_df = scraped_df.iloc[:, :4].copy()
scraped_df.columns = ["country_or_region", "currency", "symbol", "iso_code"]

scraped_df.head()

,country_or_region,currency,symbol,iso_code
0,Abkhazia,Abkhazian apsar[E],аԥ,(none)
1,Russian ruble,₽,RUB,Kopeck
2,Afghanistan,Afghan afghani,؋‎,AFN
3,Akrotiri and Dhekelia,Euro,€,EUR
4,Albania,Albanian lek,L,ALL


### Scraped Table Validation

The scraped currency table is checked for required columns, row count, missing values, and text fields.

In [23]:
required_scrape_columns = ["country_or_region", "currency", "symbol", "iso_code"]

print("Shape:", scraped_df.shape)
print("Required columns present:", all(col in scraped_df.columns for col in required_scrape_columns))
print("Missing values:")
print(scraped_df[required_scrape_columns].isna().sum())

print("Column types:")
print(scraped_df.dtypes)

Shape: (260, 4)
Required columns present: True
Missing values:
country_or_region    0
currency             0
symbol               0
iso_code             0
dtype: int64
Column types:
country_or_region    object
currency             object
symbol               object
iso_code             object
dtype: object


In [24]:
scrape_output_path = DATA_RAW / f"scrape_wikipedia_currencies_{timestamp}.csv"

scraped_df.to_csv(scrape_output_path, index=False)

print("Saved scraped data to:", scrape_output_path)

Saved scraped data to: /Users/devampatel/Documents/NYU/homework/homework04/data/raw/scrape_wikipedia_currencies_20260820-2242.csv


## Sources, Validation, Assumptions, and Risks

### Sources

API source: Frankfurter foreign exchange API  
URL: https://api.frankfurter.app/latest

Scraped source: Wikipedia list of circulating currencies  
URL: https://en.wikipedia.org/wiki/List_of_circulating_currencies

### Validation Logic

For the API dataset, I checked that the required columns were present, counted missing values, confirmed the shape, and converted exchange rates to numeric values.

For the scraped dataset, I checked that the expected columns were present, counted missing values, confirmed the shape, and reviewed the column data types.

### Assumptions and Risks

The API pull assumes that the Frankfurter endpoint is available and returns the expected exchange-rate fields.

The scraping step assumes that the Wikipedia table structure stays similar. If the page layout changes, the parser may need to be updated.

Both datasets are saved as raw CSV files so that the original acquired data is preserved before later cleaning or transformation.